In [ ]:
import sys, subprocess, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.algorithms.classifiers import QSVC
from qiskit_machine_learning.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [ ]:
# ===== Parametry do łatwej zmiany =====
data_path   = "countsAll_fixed_07_07_23.csv"  # ścieżka do pliku z danymi
sep         = "\t"                            # separator (w Twoim pliku jest tab)
n_components_pca = 2                          # liczba komponentów PCA = liczba kubitów
test_size   = 0.20                            # ułamek danych do testu
random_state = 42                             # ziarno losowe
maxiter     = 10                             # iteracje optymalizatora
entanglement = "linear"                       # "linear" | "full" | lista par
reps_feature = 2                              # głębokość feature map
reps_ansatz  = 2  

In [ ]:
df = pd.read_csv(data_path, sep=sep)
df = df.T

In [ ]:
metadata = pd.read_csv("SampleInfo_fixed_08_07_23.csv", delimiter=";")
metadata = metadata.set_index("id")
metadata["label"] = metadata["GroupAlternative"].apply(
    lambda x: 0 if x == "Asymptomatic controls" else (1 if x == "Non-small-cell lung cancer" else np.nan)
)
metadata = metadata[metadata["RealLocation"] != "Institute 5"]
df = df.merge(metadata, left_index=True, right_index=True)

In [ ]:
X = df.drop(columns=metadata.columns)
y = df["label"]

In [ ]:
df_full_clean = df.dropna(subset=["label"])
X = df_full_clean.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")
y = df_full_clean["label"]
print("Rozkład klas po czyszczeniu:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

In [ ]:
# Najpierw wybierz 100 najlepiej różnicujących cech
selector = SelectKBest(score_func=f_classif, k=100)
X_selected = selector.fit_transform(X, y)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std  = scaler.transform(X_test)

# Dopiero potem licz PCA na tych cechach
pca = PCA(n_components=n_components_pca, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca  = pca.transform(X_test_std)
print(f"Po PCA: X_train = {X_train_pca.shape}, X_test = {X_test_pca.shape}")

In [ ]:
feature_map = ZZFeatureMap(feature_dimension=n_components_pca, reps=2, entanglement="linear")

sampler = Sampler()

fidelity = ComputeUncompute(sampler=sampler)

kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

In [ ]:
t0 = time.time()
qsvc = QSVC(quantum_kernel=kernel)
qsvc.fit(X_train_pca, y_train)
y_pred = qsvc.predict(X_test_pca)
acc_qsvc = accuracy_score(y_test, y_pred)
qsvc_score = qsvc.score(X_train_pca, y_train)
t_qsvc = time.time() - t0
print(f"QSVC classification test score: {qsvc_score:.4f}")
print(f"Accuracy: {acc_qsvc:.4f}")
print(f"Czas wykonania (s): {t_qsvc:.2f}")